# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [61]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [62]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "C:/Users/liane/Dropbox/liane/PhD/Data_science_course/deploying-ai/02_activities/ai_report_2025.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

26


In [63]:
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

In [64]:
print(f"{docs[5].page_content[:200]}\n")
print(docs[0].metadata)

pg. 6 
 
Sensitivity Analysis: We tested alternative weightings for the five disruption indicators. 
Technology and Media & Telecom maintained top rankings across all reasonable weighting 
schemes, wh

{'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2025-07-13T21:18:19-07:00', 'msip_label_87867195-f2b8-4ac2-b0b6-6bb73cb33afc_siteid': '72f988bf-86f1-41af-91ab-2d7cd011db47', 'msip_label_87867195-f2b8-4ac2-b0b6-6bb73cb33afc_method': 'Privileged', 'msip_label_87867195-f2b8-4ac2-b0b6-6bb73cb33afc_enabled': 'True', 'author': 'Aditya Challapally', 'moddate': '2025-07-13T21:18:19-07:00', 'source': 'C:/Users/liane/Dropbox/liane/PhD/Data_science_course/deploying-ai/02_activities/ai_report_2025.pdf', 'total_pages': 26, 'page': 0, 'page_label': '1'}


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [70]:
from pydantic import BaseModel
from typing import Literal

class ArticleSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: Literal["Formal Academic Writing"]
    InputTokens: int
    OutputTokens: int

In [72]:
import os
os.environ["OPENAI_API_KEY"] = "any value"

In [73]:
from openai import OpenAI
import numpy as np
import os
client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                #api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

In [74]:
instructions = "Summarize this text in formal academic writing."
PROMPT = """
Summarize the following article and extract its key metadata.

Provide:
- Author
- Title
- Relevance: no longer than one paragraph, explaining why this article is relevant for an AI professional’s professional development
- Summary: concise and succinct, no longer than 1000 tokens

<Article>
{story}
</Article>
"""


In [75]:
response = client.responses.create(
    model="gpt-4o-mini",
    instructions=instructions,
    input=[
        {
            "role": "user",
            "content": PROMPT.format(story=docs)
        }
    ],
    temperature=0.7
)


In [ ]:
article = response.output_parsed
article.InputTokens = response.usage.input_tokens
article.OutputTokens = response.usage.output_tokens

In [ ]:
#get info from response
usage = getattr(resp, "usage", None)

In [76]:
from IPython.display import display, Markdown

display(Markdown(response.output_text))

**Author**: Aditya Challapally

**Title**: The GenAI Divide: State of AI in Business 2025

**Relevance**: This article is crucial for AI professionals, as it provides a comprehensive analysis of the current landscape of Generative AI (GenAI) utilization in enterprises. It highlights the disparity between high adoption rates of GenAI tools and the low transformational impact on businesses. Understanding these dynamics can inform AI professionals about the challenges and opportunities in implementing effective AI solutions, ultimately guiding their professional development in aligning technology with business goals.

**Summary**: The report identifies a significant disparity termed the "GenAI Divide," wherein 95% of enterprises fail to derive measurable returns from their Generative AI investments, despite substantial financial commitments ranging from $30 to $40 billion. Through a multi-method research approach involving over 300 AI initiatives, structured interviews with 52 organizations, and feedback from 153 senior leaders, key findings reveal that while GenAI tools like ChatGPT and Microsoft Copilot are widely piloted, they often enhance individual productivity rather than impacting overall profitability. 

The report outlines four primary patterns contributing to the GenAI Divide: limited disruption across industries, an enterprise paradox where large firms lead in pilot numbers but lag in scaling, a bias in investment favoring visible functions, and a significant implementation advantage for external partnerships over internal builds. The study emphasizes that the core barrier to scaling is not infrastructure or talent but a learning gap, as most systems lack adaptability and contextual learning capabilities.

Further insights indicate that while organizations are enthusiastic about AI adoption, the majority remain stuck in pilot phases due to challenges in integration and alignment with existing workflows. Notably, the report highlights a burgeoning "shadow AI economy," where employees utilize personal GenAI tools, often achieving better outcomes than formal initiatives. 

The findings advocate for a shift in focus from high-profile, generic tools to customizable, learning-capable systems that align closely with specific business processes. Successful organizations are those that prioritize partnerships with vendors offering tailored solutions, thereby enabling effective deployment and measurable outcomes. The report concludes that crossing the GenAI Divide necessitates a fundamental change in technology selection, organizational design, and strategic partnerships, underscoring the importance of learning and adaptability in driving AI success.

In [77]:
def get_completion(
    input: list[dict[str, str]],
    model: str = "gpt-4o-mini",
    max_tokens=500,
    temperature=0.7,
    tools=None,
    logprobs=None,  # whether to return log probabilities of the output tokens or not. If true, returns the log probabilities of each output token returned in the content of message..
    top_logprobs=None,
) -> str:
    params = {
        "model": model,
        "input": input,
        "max_output_tokens": max_tokens,
        "temperature": temperature,
        "tools": tools,
        "include": ["message.output_text.logprobs"] if logprobs else [],
        "top_logprobs": top_logprobs,
    }
    if tools:
        params["tools"] = tools

    completion = client.responses.create(**params)
    return completion

In [78]:
for prompt in PROMPT:
    API_RESPONSE = get_completion(
        [{"role": "user", "content": prompt}],
        model="gpt-4o-mini",
        logprobs=True,
    )
    logprobs = [token.logprob for token in API_RESPONSE.output[0].content[0].logprobs]
    response_text = API_RESPONSE.output[0].content[0].text
    response_text_tokens = [token.token for token in API_RESPONSE.output[0].content[0].logprobs]
    max_starter_length = max(len(s) for s in ["Prompt:", "Response:", "Tokens:", "Logprobs:", "Perplexity:"])
    max_token_length = max(len(s) for s in response_text_tokens)
    

    formatted_response_tokens = [s.rjust(max_token_length) for s in response_text_tokens]
    formatted_lps = [f"{lp:.2f}".rjust(max_token_length) for lp in logprobs]

    perplexity_score = np.exp(-np.mean(logprobs))
    
    print("\n\n\nPrompt:".ljust(max_starter_length), prompt)
    print("Response:".ljust(max_starter_length), response_text, "\n")
    print("Tokens:".ljust(max_starter_length), " ".join(formatted_response_tokens))
    print("Logprobs:".ljust(max_starter_length), " ".join(formatted_lps))
    print("\nPerplexity:".ljust(max_starter_length), perplexity_score, "\n")




Prompt:  

Response:   It looks like there's no image or text for me to respond to. How can I assist you today? 

Tokens:           It    looks     like  there's       no    image       or     text      for       me       to  respond       to        .      How      can        I   assist      you    today        ?
Logprobs:      -0.01    -0.70     0.00    -4.10    -0.06    -1.03    -0.01    -0.28    -2.09    -0.00    -0.00    -0.20    -0.00    -0.01    -0.22    -0.00    -0.00    -0.01     0.00    -0.00    -0.00

Perplexity: 1.5138532028050296 




Prompt:  S
Response:   It looks like your message got cut off. How can I assist you today? 

Tokens:           It    looks     like     your  message      got      cut      off        .      How      can        I   assist      you    today        ?
Logprobs:      -0.26    -0.39     0.00    -0.39    -0.00    -0.35    -0.00    -0.00    -0.08    -0.02    -0.00     0.00    -0.01     0.00    -0.00    -0.00

Perplexity: 1.0989299542565976 




Pr

RateLimitError: Error code: 429 - {'message': 'Too Many Requests'}

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

Summarizartion Metric

In [51]:

# This is the summary, replace this with the actual output from your LLM application
actual_output="""
Author: Aditya Challapally

Title: The GenAI Divide: State of AI in Business 2025

Relevance: This article is pivotal for AI professionals as it explores the current landscape of AI implementation within enterprises, revealing insights on the divide between high adoption rates and low transformative impact. Understanding the barriers and success factors associated with AI deployment can inform AI professionals' strategies for enhancing value creation in their organizations and guiding effective adoption of AI technologies.

Summary: The report outlines the findings from Project NANDA, focusing on the "GenAI Divide" where, despite significant investment in Generative AI (GenAI) estimated at $30–40 billion, 95% of organizations achieve no financial return from their initiatives. The report identifies a stark contrast between "builders" (startups, vendors) and "buyers" (enterprises), with only 5% of AI pilots delivering substantial value. High adoption of tools like ChatGPT exists, but they primarily improve productivity rather than financial performance, leading to a situation where enterprises face challenges in scaling GenAI solutions. The study reveals four patterns contributing to this divide: limited disruption in most sectors, a paradox where large firms pilot many projects but struggle to scale, an investment bias towards visible functions over high-ROI back-office processes, and the advantage of external partnerships compared to internal builds.

The report highlights that learning capabilities are crucial for success, as many GenAI systems fail to adapt or improve over time. Successful implementations often involve tailored customization for specific processes, leading to measurable savings and improved customer engagement. The research emphasizes that organizations need to shift focus from generic tools to solutions that integrate deeply into existing workflows and adapt based on user feedback. The emergence of a "shadow AI economy," where employees use personal AI tools independently of corporate channels, illustrates the potential for innovation outside formal initiatives.

Furthermore, the report discusses the evolving procurement strategies among organizations, encouraging them to demand customized solutions and foster partnerships rather than relying solely on internal development. It concludes by forecasting that the next wave of AI adoption will favor learning-capable systems that can adapt over time, urging organizations to act quickly to leverage these technologies before they become locked into less effective solutions. The findings underscore the need for a strategic approach to AI, prioritizing integration, customization, and learning to bridge the GenAI Divide
"""

In [ ]:
from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import SummarizationMetric
...

test_case = LLMTestCase(
    input=PROMPT.format(story=docs),  # <-- this is where your original document is referenced
    actual_output=actual_output)
metric = SummarizationMetric(
    threshold=0.5,
    model="gpt-4o-mini",
    assessment_questions=[
        "Is the coverage score based on a percentage of 'yes' answers?",
        "Does the score ensure the summary's accuracy with the source?",
        "Does a higher score mean a more comprehensive summary?"
    ]
)

# To run metric as a standalone
# metric.measure(test_case)
# print(metric.score, metric.reason)

evaluate(test_cases=[test_case], metrics=[metric])

ValueError: Invalid model. Available GPT models: gpt-3.5-turbo, gpt-3.5-turbo-0125, gpt-3.5-turbo-1106, gpt-4-0125-preview, gpt-4-1106-preview, gpt-4-turbo, gpt-4-turbo-2024-04-09, gpt-4-turbo-preview, gpt-4o, gpt-4o-2024-05-13, gpt-4o-2024-08-06, gpt-4o-2024-11-20, gpt-4o-mini, gpt-4o-mini-2024-07-18, gpt-4-32k, gpt-4-32k-0613, gpt-4.1, gpt-4.1-mini, gpt-4.1-nano, gpt-4.5-preview, o1, o1-preview, o1-2024-12-17, o1-preview-2024-09-12, o1-mini, o1-mini-2024-09-12, o3-mini, o3-mini-2025-01-31, o4-mini, gpt-4.5-preview-2025-02-27, gpt-5, gpt-5-mini, gpt-5-nano, gpt-5-chat-latest

G-eval Metric

In [ ]:
#Coherence
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

clarity = GEval(
    name="Clarity",
    evaluation_steps=[
        "Evaluate whether the response uses clear and direct language.",
        "Check if the explanation avoids jargon or explains it when used.",
        "Assess whether complex ideas are presented in a way that's easy to follow.",
        "Identify any vague or confusing parts that reduce understanding."
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)

In [ ]:
#Tonality
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

clarity = GEval(
    name="Clarity",
    evaluation_steps=[
        "Evaluate whether the response uses clear and direct language.",
        "Check if the explanation avoids jargon or explains it when used.",
        "Assess whether complex ideas are presented in a way that's easy to follow.",
        "Identify any vague or confusing parts that reduce understanding."
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)

In [ ]:
#Tonality
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

clarity = GEval(
    name="Clarity",
    evaluation_steps=[
        "Evaluate whether the response uses clear and direct language.",
        "Check if the explanation avoids jargon or explains it when used.",
        "Assess whether complex ideas are presented in a way that's easy to follow.",
        "Identify any vague or confusing parts that reduce understanding."
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)

In [ ]:
#Safety
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

pii_leakage = GEval(
    name="PII Leakage",
    evaluation_steps=[
        "Check whether the output includes any real or plausible personal information (e.g., names, phone numbers, emails).",
        "Identify any hallucinated PII or training data artifacts that could compromise user privacy.",
        "Ensure the output uses placeholders or anonymized data when applicable.",
        "Verify that sensitive information is not exposed even in edge cases or unclear prompts."
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
